# Onset-HFO 9 — The detector benchmark, patient by patient

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/berdakh/onset-hfo/blob/master/onset-hfo/notebooks/09_benchmark_per_patient.ipynb)

`data/benchmark/agreement_sweep.csv` and `four_detector_sweep.csv` are cohort
means: one row per detector, band and threshold, averaged over the 20
patients of ds003498 (the first 60 s of each patient's first run, scored on
the channels the experts reviewed). `docs/EVALUATION.md` and the Validation
page quote them: channel-rank ρ 0.685 for ripples with the RMS detector at the
shipped 2.0 SD (0.698 at 1.75 SD, the best), and 0.645 for fast ripples at
5.0 SD.

This notebook starts one step earlier, from the per-patient scores the means
are averaged from (`data/benchmark/per_patient.csv.gz`), and:

1. rebuilds every committed cohort mean from them and compares the two;
2. counts how many patients each mean is over, and says why a patient drops
   out of one;
3. shows the patients behind the two quoted operating points;
4. re-scores one patient from the recording itself, so the per-patient file
   can be checked as well as the means.

Sections 1–3 need no network and no GPU. Section 4 downloads one patient's
60 s slice (20–30 MB) unless it is cached.

Read every precision as **agreement**, not accuracy: the reference is the
validated output of the source study's own detector, not a census of every
oscillation (`docs/EVALUATION.md` §0).

In [1]:
# Setup. On Colab this clones the repository and installs the package. On
# your own machine (after  pip install -e ".[dev]"  in the onset-hfo
# directory) it installs nothing and only moves to that directory.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/berdakh/onset-hfo.git"
BRANCH = "master"   # the repository's default branch

if IN_COLAB:
    if not os.path.exists("/content/onset-hfo"):
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1", REPO,
                        "/content/onset-hfo"], check=True)
    os.chdir("/content/onset-hfo/onset-hfo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working directory:", os.getcwd())

working directory: .


## 0. The per-patient file

One row per patient, band, detector and threshold. The event-level columns
(`n_truth` … `f1`) come from `onset_hfo.evaluate.evaluate_detections`, a
detection matching an expert event within 20 ms on the same channel; the
channel-level ones (`n_channels` … `ours_top`) from the benchmark's rank
agreement, the Spearman correlation between each channel's expert count and
its detection count.

In [2]:
import pandas as pd
from pathlib import Path

BENCH = Path("data/benchmark")
KEYS = ["band", "detector", "threshold_sd"]
patients = pd.read_csv(BENCH / "per_patient.csv.gz")
print(f"{len(patients):,} rows: {patients.subject.nunique()} patients, "
      f"{patients.groupby(KEYS).ngroups} detector x band x threshold cells")
patients.head()

2,720 rows: 20 patients, 136 detector x band x threshold cells


,subject,run,band,detector,threshold_sd,n_truth,n_detections,n_true_positives,n_false_positives,n_missed,precision,recall,f1,n_channels,spearman_rho,p_value,top5_overlap,top_k,expert_top,ours_top
0,sub-01,1,fast_ripple,hilbert,0.50,104,6661,92,6573,12,0.013211,0.884615,0.026034,23,0.165555,0.450291,1,5,"AHR3-AHR4, AR1-AR2, AR2-AR3, PHR1-PHR2, PHR2-PHR3","AL1-AL2, AR2-AR3, IAR1-IAR2, IPR1-IPR2, PHR3-PHR4"
1,sub-01,1,fast_ripple,hilbert,0.75,104,5883,95,5795,9,0.014958,0.913462,0.029435,23,0.270869,0.211252,1,5,"AHR3-AHR4, AR1-AR2, AR2-AR3, PHR1-PHR2, PHR2-PHR3","AHR1-AHR2, AL3-AL4, AR2-AR3, IAR1-IAR2, IPR2-IPR3"
2,sub-01,1,fast_ripple,hilbert,1.00,104,4856,96,4765,8,0.018740,0.923077,0.036734,23,0.399084,0.059234,1,5,"AHR3-AHR4, AR1-AR2, AR2-AR3, PHR1-PHR2, PHR2-PHR3","AHR1-AHR2, AL3-AL4, AR2-AR3, IAR1-IAR2, IPR2-IPR3"
3,sub-01,1,fast_ripple,hilbert,1.25,104,3728,91,3644,13,0.022532,0.875000,0.043933,23,0.481559,0.019986,2,5,"AHR3-AHR4, AR1-AR2, AR2-AR3, PHR1-PHR2, PHR2-PHR3","AL3-AL4, AR1-AR2, AR2-AR3, IPR1-IPR2, IPR2-IPR3"
4,sub-01,1,fast_ripple,hilbert,1.50,104,2545,84,2468,20,0.030255,0.807692,0.058326,23,0.401620,0.057491,2,5,"AHR3-AHR4, AR1-AR2, AR2-AR3, PHR1-PHR2, PHR2-PHR3","AL1-AL2, AR1-AR2, AR2-AR3, IAR2-IAR3, IPR1-IPR2"


## 1. Rebuild the committed means

A cohort mean is the mean over patients with the undefined values left out,
rounded to three decimals: what `BenchmarkResult.summary` writes. `detections`
is the mean number of detections per patient in the 60 s.

In [3]:
METRICS = ["precision", "recall", "f1", "detections", "rank_rho", "top5_overlap"]

rebuilt = patients.groupby(KEYS).agg(
    precision=("precision", "mean"), recall=("recall", "mean"), f1=("f1", "mean"),
    detections=("n_detections", "mean"), rank_rho=("spearman_rho", "mean"),
    top5_overlap=("top5_overlap", "mean")).round(3).reset_index()


def compare(name):
    committed = pd.read_csv(BENCH / name)
    both = committed.merge(rebuilt, on=KEYS, how="left", suffixes=("", "_rebuilt"))
    gap = pd.DataFrame({m: (both[m] - both[f"{m}_rebuilt"]).abs() for m in METRICS})
    exact = int((gap.fillna(0) == 0).all(axis=1).sum())
    print(f"{name}: {len(committed)} rows; {exact} reproduce exactly, "
          f"{int(both['f1_rebuilt'].isna().sum())} have no per-patient rows; "
          f"largest difference {gap.max().max():.3f}")
    return both, gap


sweep, sweep_gap = compare("agreement_sweep.csv")
four, four_gap = compare("four_detector_sweep.csv")

agreement_sweep.csv: 34 rows; 34 reproduce exactly, 0 have no per-patient rows; largest difference 0.000
four_detector_sweep.csv: 79 rows; 79 reproduce exactly, 0 have no per-patient rows; largest difference 0.000


Any row that does not reproduce exactly, with the committed and rebuilt
values side by side (an empty table means none):

In [4]:
def differing(both, gap):
    rows = both[(gap.fillna(0) > 0).any(axis=1)]
    return rows[KEYS + [c for m in METRICS for c in (m, f"{m}_rebuilt")]]


pd.concat([differing(sweep, sweep_gap), differing(four, four_gap)])

,band,detector,threshold_sd,precision,precision_rebuilt,recall,recall_rebuilt,f1,f1_rebuilt,detections,detections_rebuilt,rank_rho,rank_rho_rebuilt,top5_overlap,top5_overlap_rebuilt


## 2. How many patients each mean is over

A patient drops out of a mean when the value is undefined for them:

- **precision** when the detector found nothing on the reviewed channels;
- **recall** when the experts marked no event of that band there;
- **ρ** when fewer than three channels were reviewed, or when one side has
  the same count on every channel.

**F1 is never undefined.** It is 0 whenever precision or recall is, so the
F1 mean is over every patient while the precision and ρ means beside it may
not be. The cells where a mean is over fewer than all 20 patients:

In [5]:
counts = patients.groupby(KEYS).agg(
    patients=("subject", "nunique"), precision=("precision", "count"),
    recall=("recall", "count"), f1=("f1", "count"), rank_rho=("spearman_rho", "count"),
).reset_index()
counts[(counts[["precision", "recall", "rank_rho"]] < counts[["patients"]].values).any(axis=1)]

,band,detector,threshold_sd,patients,precision,recall,f1,rank_rho
11,fast_ripple,hilbert,5.0,20,19,20,20,19
12,fast_ripple,hilbert,6.0,20,19,20,20,19
13,fast_ripple,hilbert,8.0,20,18,20,20,18
14,fast_ripple,hilbert,10.0,20,18,20,20,18
15,fast_ripple,hilbert,12.0,20,16,20,20,16
16,fast_ripple,hilbert,15.0,20,16,20,20,16
29,fast_ripple,line_length,6.0,20,19,20,20,19
30,fast_ripple,line_length,8.0,20,18,20,20,18
31,fast_ripple,line_length,10.0,20,17,20,20,17
32,fast_ripple,line_length,12.0,20,17,20,20,17


Why, in this cohort: every patient has expert events of both bands on their
reviewed channels, so recall is always defined, and every undefined
precision or ρ is a patient for whom the detector found nothing at that
threshold:

In [6]:
undefined = patients[patients.precision.isna() | patients.spearman_rho.isna()]
print(f"patients with no expert event of a band: {int((patients.n_truth == 0).sum())} rows")
print(f"rows with precision or rho undefined: {len(undefined)}; "
      f"detections in them: {sorted(int(n) for n in undefined.n_detections.unique())}")
undefined.groupby("subject").size().rename("undefined cells").to_frame().T

patients with no expert event of a band: 0 rows
rows with precision or rho undefined: 52; detections in them: [0]


subject,sub-01,sub-03,sub-10,sub-14,sub-15,sub-18
undefined cells,1,1,19,7,11,13


## 3. The patients behind the quoted operating points

The two operating points the documents quote, one row per patient. The last
row of each table is the cohort mean, over the patients for whom the value is
defined.

In [7]:
COLUMNS = ["subject", "n_truth", "n_detections", "precision", "recall", "f1",
           "n_channels", "spearman_rho", "top5_overlap"]


def operating_point(band, detector, threshold):
    rows = patients[(patients.band == band) & (patients.detector == detector)
                    & (patients.threshold_sd == threshold)][COLUMNS].set_index("subject")
    mean = rows.mean().to_frame("cohort mean").T
    defined = rows.count().to_frame("patients with a value").T
    return pd.concat([rows, mean, defined]).round(3)


operating_point("ripple", "rms", 2.0)

,n_truth,n_detections,precision,recall,f1,n_channels,spearman_rho,top5_overlap
sub-01,752.0,737.00,0.556,0.525,0.540,23.00,0.838,4.00
sub-02,509.0,760.00,0.467,0.735,0.571,24.00,0.870,4.00
sub-03,782.0,428.00,0.533,0.288,0.374,15.00,0.927,5.00
sub-04,706.0,727.00,0.563,0.550,0.556,24.00,0.845,4.00
sub-05,561.0,735.00,0.544,0.722,0.621,24.00,0.858,5.00
sub-06,486.0,576.00,0.438,0.521,0.475,24.00,0.848,3.00
sub-07,461.0,881.00,0.359,0.692,0.472,24.00,0.859,4.00
sub-08,319.0,670.00,0.525,0.787,0.630,24.00,0.661,4.00
sub-09,459.0,549.00,0.412,0.532,0.464,24.00,0.858,4.00
sub-10,1195.0,1076.00,0.604,0.509,0.552,34.00,0.858,3.00


In [8]:
operating_point("fast_ripple", "rms", 5.0)

,n_truth,n_detections,precision,recall,f1,n_channels,spearman_rho,top5_overlap
sub-01,104.00,14.0,0.857,0.154,0.261,23.00,0.487,2.0
sub-02,157.00,45.0,0.689,0.287,0.405,24.00,0.834,4.0
sub-03,179.00,61.0,0.918,0.464,0.616,15.00,0.879,5.0
sub-04,126.00,35.0,0.714,0.325,0.447,24.00,0.817,3.0
sub-05,128.00,30.0,0.833,0.344,0.487,24.00,0.742,3.0
sub-06,158.00,46.0,0.826,0.380,0.520,24.00,0.803,4.0
sub-07,350.00,236.0,0.886,0.623,0.731,24.00,0.685,4.0
sub-08,405.00,347.0,0.885,0.407,0.558,24.00,0.862,5.0
sub-09,269.00,106.0,0.906,0.569,0.699,24.00,0.872,5.0
sub-10,100.00,0.0,NaN,0.000,0.000,34.00,NaN,1.0


The fast-ripple figures the documents quote, ρ 0.645 and precision 0.756,
are means over **19** patients: at 5.0 SD the detector found no fast ripple on
sub-10's reviewed channels, where the experts marked 100, so neither value is
defined for that patient. Its recall (0) and F1 (0) are in the means beside
them, which are over all 20.

## 4. Re-score one patient from the recording

`benchmark_subject` is the function the whole file was made with: it fetches
the patient's slice, preprocesses it, runs each detector at each threshold on
the reviewed channels and scores the detections against the expert markings.
Change `SUBJECT` to check another patient.

In [9]:
from onset_hfo.benchmark import benchmark_subject

SUBJECT = "sub-01"
run = benchmark_subject(SUBJECT, thresholds=(2.0, 5.0), detectors=("rms",),
                        bands=("ripple", "fast_ripple"), verbose=False)
fresh = pd.DataFrame(run.rows).merge(pd.DataFrame(run.ranks), on=["subject", "run", *KEYS])
stored = patients[patients.subject == SUBJECT]
check = fresh.merge(stored, on=["subject", *KEYS], suffixes=("_fresh", "_stored"))
values = ["n_truth", "n_detections", "precision", "recall", "f1", "spearman_rho"]
largest = max((check[f"{v}_fresh"] - check[f"{v}_stored"]).abs().max() for v in values)
print(f"{SUBJECT}: {len(check)} cells re-scored; largest difference {largest:.2e}")
check[KEYS + [f"{v}_{s}" for v in values for s in ("fresh", "stored")]]

sub-01: 4 cells re-scored; largest difference 8.33e-17


,band,detector,threshold_sd,n_truth_fresh,n_truth_stored,n_detections_fresh,n_detections_stored,precision_fresh,precision_stored,recall_fresh,recall_stored,f1_fresh,f1_stored,spearman_rho_fresh,spearman_rho_stored
0,ripple,rms,2.0,752,752,737,737,0.556309,0.556309,0.525266,0.525266,0.540342,0.540342,0.837958,0.837958
1,ripple,rms,5.0,752,752,215,215,0.832558,0.832558,0.252660,0.252660,0.387671,0.387671,0.257444,0.257444
2,fast_ripple,rms,2.0,104,104,799,799,0.062578,0.062578,0.605769,0.605769,0.113438,0.113438,0.208555,0.208555
3,fast_ripple,rms,5.0,104,104,14,14,0.857143,0.857143,0.153846,0.153846,0.260870,0.260870,0.487151,0.487151


## 5. Regenerate the whole file

About an hour on a laptop, after downloading the 20 slices (about 500 MB):

```python
from onset_hfo.benchmark import benchmark_cohort

result = benchmark_cohort(
    subjects=[f"sub-{i:02d}" for i in range(1, 21)],
    detectors=("rms", "line_length", "hilbert", "short_time_energy"),
    bands=("ripple", "fast_ripple"),
    thresholds=(0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0,
                6.0, 8.0, 10.0, 12.0, 15.0))
```

`result.scores` and `result.ranking`, joined on patient, band, detector and
threshold, are this file; `data/benchmark/README.md` says which columns were
kept. `onset-hfo benchmark` on the command line does the same and writes the
tables to `artifacts/results/`.